In [1]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq    
from langchain_tavily import TavilySearch
from langchain.tools import tool
import requests

In [2]:
from langchain.agents import create_agent

In [3]:
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")
os.environ["WEATHERSTACK_API_KEY"] = os.getenv("WEATHERSTACK_API_KEY")

In [4]:
search_tool = TavilySearch(max_results=3)

In [5]:
@tool
def get_weather(city: str) -> str:
    """Fetch  current weather information for a city"""

    api_key = os.getenv("WEATHERSTACK_API_KEY")

    if not api_key:
        return "WEATHERSTACK_API_KEY is not set"

    url = (
        f"http://api.weatherstack.com/current"
        f"?access_key={api_key}"
        f"&query={city}"
    )

    response = requests.get(url)
    data = response.json()

    if "current" not in data:
        return f"Could not fetch weather data for {city}: {data}"

    return (
        f"City: {city}\n"
        f"Temperature: {data['current']['temperature']}°C\n"
        f"Weather: {data['current']['weather_descriptions'][0]}\n"
        f"Humidity: {data['current']['humidity']}%"
    )

In [6]:
result = search_tool.invoke("What is the latest news on AI?")
result

{'query': 'What is the latest news on AI?',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://www.artificialintelligence-news.com',
   'title': 'AI News | Latest News | Insights Powering AI-Driven Business Growth',
   'content': 'October 30, 2025\n\n### Malaysia launches Ryt Bank, its first AI-powered bank\n\nFinance AI\n\nAugust 26, 2025\n\n### Google’s Veo 3 AI video creation tools are now widely available\n\nAI in Action\n\nJuly 29, 2025\n\n#### Computer Vision\n\n### Microsoft’s Majorana 2 quantum chip is also a case study for agentic AI in R&D\n\nInside AI\n\nJune 3, 2026\n\n### US and Japan announce sweeping AI and tech collaboration\n\nArtificial Intelligence\n\nApril 11, 2024\n\n### UK and Canada sign AI compute agreement [...] August 18, 2026\n\n### Okta targets AI agent token costs with MCP scoping\n\nData Engineering & MLOps\n\nAugust 13, 2026\n\n### Red Hat, NVIDIA, IBM back project turning AI policy into code\n\nGovernance, Regulat

In [7]:
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    api_key = os.environ["GROQ_API_KEY"]
)

In [8]:
response = llm.invoke("What year is it?")
response

AIMessage(content='It’s 2026.', additional_kwargs={'reasoning_content': 'The user asks: "What year is it?" The current date is 2026-09-23. So answer: 2026. Probably mention that it\'s 2026.'}, response_metadata={'token_usage': {'completion_tokens': 53, 'prompt_tokens': 76, 'total_tokens': 129, 'completion_time': 0.05924616, 'completion_tokens_details': {'reasoning_tokens': 38}, 'prompt_time': 0.01767812, 'prompt_tokens_details': None, 'queue_time': 0.344137424, 'total_time': 0.07692428}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_63473661d7', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0cdc0-ca12-7703-a6e2-990949e01c32-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 76, 'output_tokens': 53, 'total_tokens': 129, 'output_token_details': {'reasoning': 38}})

In [9]:
# tools = []
tools = [search_tool, get_weather]

In [10]:
# Create Agent
agent = create_agent(
    model = llm,
    tools = tools
)

In [11]:
response = agent.invoke({
    "messages": [
        {"role": "user", "content": "Find the current weather in Ongole"}
    ]
})
response

{'messages': [HumanMessage(content='Find the current weather in Ongole', additional_kwargs={}, response_metadata={}, id='1e920e9a-28a9-4bb7-9cd2-843c2ec6c0d3'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to get current weather in Ongole. Use get_weather.', 'tool_calls': [{'id': 'fc_b3e7fac9-cd24-4326-bd7b-63d357fb3679', 'function': {'arguments': '{"city":"Ongole"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 40, 'prompt_tokens': 344, 'total_tokens': 384, 'completion_time': 0.05631404, 'completion_tokens_details': {'reasoning_tokens': 15}, 'prompt_time': 0.021030001, 'prompt_tokens_details': None, 'queue_time': 0.335386794, 'total_time': 0.077344041}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_1074f9ce08', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0cdc0-cf17-72f0-bada-0d0611310da5-0', tool_calls=[{'name

In [12]:
print(response["messages"][-1].content)

**Current Weather in Ongole**

- **Temperature:** 25 °C  
- **Weather:** Patchy rain nearby  
- **Humidity:** 88%


In [13]:
for msg in response["messages"]:
    print(msg.content)

Find the current weather in Ongole

City: Ongole
Temperature: 25°C
Weather: Patchy rain nearby
Humidity: 88%
**Current Weather in Ongole**

- **Temperature:** 25 °C  
- **Weather:** Patchy rain nearby  
- **Humidity:** 88%


In [14]:
response = agent.invoke({
    "messages": [
        {"role": "user", "content": "Tell me about the Latest News on Iran and US War"}
    ]
})

In [15]:
for msg in response["messages"]:
    print(msg.content)

Tell me about the Latest News on Iran and US War

{"query": "Iran US war latest news 2026", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://www.aljazeera.com/news/liveblog/2026/9/11/iran-war-live-irgc-attacks-us-unmanned-vessel-in-hormuz-two-ships-hit", "title": "Iran war live: IRGC attacks US unmanned vessel in Hormuz; two ships hit | Donald Trump News | Al Jazeera", "content": "Vessels transit the Strait of Hormuz off the coast of the port city of Bandar Abbas in southern Iran on September 5, 2026. The United States said it struck and destroyed three Iranian oil tankers on September 5 in response to attempted missile attacks on naval vessels, in the latest exchange of fire since renewed hostilities flared between the foes. The new clashes, which began with US raids last week, come with both sides deadlocked in the six-month war as Iran keeps a stranglehold on the Strait of Hormuz and Washington presses its counter-blockade of Iranian ports. (Ph